### Классический генетический алгоритм. Оптимизация гиперпараметров при помощи PyGad

In [1]:
import pandas as pd
import pygad
from tqdm import tqdm
import time

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

### KNeighbors Classifier

Создадим 20 поколений, в каждом из которых будет 10 особей.

In [2]:
num_generations = 20
sol_per_pop = 10
total_evals = num_generations * sol_per_pop

gene_space = [
    {'low': 3, 'high': 30},     # n_neighbors 
    [1, 2]                   # Manhattan или Euclidean
]

Определение оценки текущей особи.

In [3]:
def fitness_func(ga_instance, solution, solution_idx):
    global pbar
    pbar.update(1)

    model = KNeighborsClassifier(
        n_neighbors=int(solution[0]),
        p=int(solution[1]),
        weights='distance',
        n_jobs=-1
    )

    score = cross_val_score(
        model,
        x_train,
        y_train,
        cv=5,
        scoring='f1_macro',
        n_jobs=-1
    ).mean()

    return score

Настройка алгоритма GA. Используем `tqdm` для просмотра прогресса.

In [4]:
n_samples = [100, 500, 1000, 3000]
m_features = [5, 8, 11]

results = []

In [5]:
for n in n_samples:
    for m in m_features:
        data = pd.read_csv(f"../classical_ml_methods/f1_data/f1_data_{n}_s_{m}_f.csv")

        pbar = tqdm(total=total_evals, desc=f"knn pygad optimization n={n} m={m}")

        y = data['collision']
        x = data.drop(['collision'], axis=1)

        global x_train, y_train
        x_train, x_test, y_train, y_test = train_test_split(
            x, y, test_size=0.2, random_state=81
            )
        
        ga_instance = pygad.GA(
            num_generations=num_generations,
            sol_per_pop=sol_per_pop,
            num_parents_mating=5,
            num_genes=2,
            fitness_func=fitness_func,
            gene_space=gene_space,
            parent_selection_type="rank",
            keep_parents=2,
            mutation_percent_genes=50,
            random_seed=81
        )

        start_time = time.time()
        ga_instance.run()
        search_time = time.time() - start_time

        pbar.close()

        solution, solution_fitness, _ = ga_instance.best_solution()
        
        results.append({
            'samples (n)': n,
            'features (m)': m,
            'best_n_neighbors': int(solution[0]),
            'best_p': int(solution[1]),
            'f1-score': solution_fitness,
            'search_time (sec)': search_time
        })

knn pygad optimization n=3000 m=11:  86%|████████▋ | 173/200 [00:05<00:00, 29.44it/s]


In [6]:
results_df = pd.DataFrame(results)
results_df

,samples (n),features (m),best_n_neighbors,best_p,f1-score,search_time (sec)
0,100,5,17,1,0.990476,6.427153
1,100,8,6,2,0.989474,4.365999
2,100,11,3,2,0.844451,4.329325
3,500,5,8,2,0.992775,4.223485
4,500,8,8,2,0.980845,4.368690
5,500,11,4,1,0.832888,4.830106
6,1000,5,5,2,0.993636,4.472729
7,1000,8,6,2,0.980838,4.233300
8,1000,11,4,1,0.837058,4.571254
9,3000,5,6,1,0.995895,4.359650


**Сравнение с предыдущими результатами:**
- `f1-score` на больших выборках увеличился
- `n_neighbors` немного изменили свои значения
- генетический алгоритм потребовал больше времени, нежели `RandomizedSearchCV`

![KNN before](knn.png)

### Сохранение модели

In [7]:
import joblib

model = KNeighborsClassifier(
    n_neighbors=int(solution[0]),
    p=int(solution[1]),
    weights='distance'
)
model.fit(x_train, y_train) 

joblib.dump(model, 'models/knn_ga_model.pkl')

['models/knn_ga_model.pkl']

### Native Bayes

In [8]:
gene_space = [
    {'low': 1e-10, 'high': 1e-7}
]

In [9]:
def fitness_func_nb(ga_instance, solution, solution_idx):
    global pbar
    pbar.update(1)

    model = GaussianNB(
        var_smoothing=solution[0]
    )

    score = cross_val_score(
        model,
        x_train,
        y_train,
        cv=5,
        scoring='f1_macro',
        n_jobs=-1
    ).mean()

    return score

In [15]:
results_nb = []

In [16]:
for n in n_samples:
    for m in m_features:
        data = pd.read_csv(f"../classical_ml_methods/f1_data/f1_data_{n}_s_{m}_f.csv")

        pbar = tqdm(total=total_evals, desc=f"native bayes pygad optimization n={n} m={m}")

        y = data['collision']
        x = data.drop(['collision'], axis=1)

        global x_train, y_train
        x_train, x_test, y_train, y_test = train_test_split(
            x, y, test_size=0.2, random_state=81
            )
        
        ga_instance = pygad.GA(
            num_generations=num_generations,
            sol_per_pop=sol_per_pop,
            num_parents_mating=5,
            num_genes=1,
            fitness_func=fitness_func_nb,
            gene_space=gene_space,
            parent_selection_type="rank",
            keep_parents=2,
            mutation_percent_genes=100,
            random_seed=81
        )

        start_time = time.time()
        ga_instance.run()
        search_time = time.time() - start_time

        pbar.close()

        solution, solution_fitness, _ = ga_instance.best_solution()
        
        results_nb.append({
            'samples (n)': n,
            'features (m)': m,
            'var_smoothing': solution[0],
            'f1-score': solution_fitness,
            'search_time (sec)': search_time
        })

native bayes pygad optimization n=3000 m=11:  95%|█████████▌| 190/200 [00:02<00:00, 76.08it/s]


In [17]:
results_nb_df = pd.DataFrame(results_nb)
results_nb_df

,samples (n),features (m),var_smoothing,f1-score,search_time (sec)
0,100,5,6.404769e-08,0.970426,2.586077
1,100,8,6.404769e-08,0.978328,2.537303
2,100,11,6.404769e-08,0.905710,2.513037
3,500,5,6.404769e-08,0.976426,2.525248
4,500,8,6.404769e-08,0.905915,2.490883
5,500,11,6.404769e-08,0.876513,2.562248
6,1000,5,6.404769e-08,0.987269,2.505840
7,1000,8,6.404769e-08,0.968370,2.552827
8,1000,11,6.404769e-08,0.871358,2.488496
9,3000,5,6.404769e-08,0.985785,2.512265


**Сравнение с предыдущими результатами:**
- `f1-score` незначительно уменьшился
- `var_smoothing` изменили свои значения
- генетический алгоритм потребовал больше времени, нежели `RandomizedSearchCV`

![NB before](native_bayes.png)

### Сохранение модели

In [19]:
model = GaussianNB(
    var_smoothing=solution[0]
)
model.fit(x_train, y_train) 

joblib.dump(model, 'models/native_bayes_ga_model.pkl')

['models/native_bayes_ga_model.pkl']